# FraudLab · protocolo híbrido 5/42

**Esta versão foi salva sem execução no Kaggle.** As células não contêm resultados de uma execução nesta plataforma. Os treinamentos locais e as submissões realmente processadas estão documentados no [repositório público e suas evidências](https://github.com/aikolopes-og/fraudlab). O ranking é uma observação datada e provisória.

O fluxo canônico treina GLM e LightGBM próprios em CV estratificada 5-fold/seed42, com configurações fixas. Combina a composição própria 75/25 com 25% de previsões públicas GPT-2 de Masaya Kawamata, sem atribuir seu treinamento a nós. Todos os rótulos participam da CV; a reserva do protocolo anterior é apenas histórica e não é uma avaliação independente desta mistura.

Código próprio MIT; previsões públicas CC BY 4.0, com os créditos abaixo. Dados da competição sob seus termos.

## 1. Fonte fixada e execução offline

Revisão [30dc7eb723e4cf31291bb075ed6c0d943557264c](https://github.com/aikolopes-og/fraudlab/tree/30dc7eb723e4cf31291bb075ed6c0d943557264c). Cada módulo abaixo é exportado com `git show` dessa revisão; os hashes conferem a integridade. Não há `pip install`, clone remoto ou execução de código de treinamento de terceiros. O notebook usa as bibliotecas já instaladas na imagem Kaggle; o preflight interrompe se forem incompatíveis. O GPT-2 entra por arrays públicos anexados como input, com versão e hashes fixados. O protocolo híbrido não exige GPU.

In [ ]:
from pathlib import Path
Path('fraudlab').mkdir(exist_ok=True)


In [ ]:
%%writefile fraudlab/__init__.py
"""Small, auditable experiments for binary tabular classification."""

__version__ = "0.1.0"


In [ ]:
%%writefile fraudlab/__main__.py
"""Run with python -m fraudlab; defaults never download or submit data."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from .data import inspect_data, validate_submission, write_json
from .experiment import blend, train


def parser() -> argparse.ArgumentParser:
    root = argparse.ArgumentParser(description="Auditable Kaggle fraud experiments.")
    commands = root.add_subparsers(dest="command", required=True)
    inspection = commands.add_parser(
        "inspect", help="Write aggregate data checks and SHA-256 fingerprints."
    )
    inspection.add_argument("--data", default="data/raw")
    inspection.add_argument("--output", default="artifacts/profile.json")
    inspection.add_argument("--id-column", default="id")
    inspection.add_argument("--target", default="is_fraud")

    training = commands.add_parser(
        "train",
        help="Fit cross-validation folds and write OOF, models and test predictions.",
    )
    training.add_argument("--data", default="data/raw")
    training.add_argument("--output", default="artifacts/runs")
    training.add_argument("--name", required=True)
    training.add_argument(
        "--phase",
        choices=("development", "exploratory_full_data_cv"),
        default="development",
        help="Label the validation protocol; exploratory full-data CV has no independent holdout.",
    )
    training.add_argument(
        "--model",
        choices=("logistic", "glm", "catboost", "lightgbm", "mlp"),
        required=True,
    )
    training.add_argument(
        "--feature-set",
        choices=("raw", "behavior", "digits", "value_digits"),
        default="raw",
    )
    training.add_argument("--id-column", default="id")
    training.add_argument("--target", default="is_fraud")
    training.add_argument(
        "--split", choices=("stratified", "group", "time"), default="stratified"
    )
    training.add_argument(
        "--validation-note",
        required=True,
        help="Explain why this split matches the data and intended generalization.",
    )
    training.add_argument("--group-column")
    training.add_argument("--time-column")
    training.add_argument("--folds", type=int, default=3)
    training.add_argument("--seed", type=int, default=42)
    training.add_argument(
        "--holdout-fraction",
        type=float,
        default=0.0,
        help="Reserve this fraction before CV; never evaluate it during model selection.",
    )
    training.add_argument("--holdout-seed", type=int, default=2026)
    training.add_argument("--threads", type=int, default=8)
    training.add_argument(
        "--fixed-training",
        action="store_true",
        help="Use the configured tree/epoch count without early stopping; useful for preselected full-data CV.",
    )
    training.add_argument(
        "--device",
        choices=("auto", "cpu", "cuda"),
        default="auto",
        help="Auto uses CPU for CatBoost and CUDA when available for the MLP.",
    )
    training.add_argument("--logistic-c", type=float, default=1.0)
    training.add_argument("--iterations", type=int, default=2000)
    training.add_argument("--depth", type=int, default=6)
    training.add_argument("--learning-rate", type=float, default=0.05)
    training.add_argument("--l2-leaf-reg", type=float, default=3.0)
    training.add_argument("--num-leaves", type=int, default=4)
    training.add_argument("--min-child-samples", type=int, default=100)
    training.add_argument("--target-smoothing", type=float, default=20.0)
    training.add_argument("--epochs", type=int, default=30)
    training.add_argument("--hidden-dims", type=int, nargs="+", default=[128, 64])
    training.add_argument("--dropout", type=float, default=0.1)
    training.add_argument("--neural-learning-rate", type=float, default=0.001)
    training.add_argument("--weight-decay", type=float, default=0.0001)
    training.add_argument(
        "--neural-numeric-transform",
        choices=("standard", "quantile"),
        default="standard",
    )
    training.add_argument("--batch-size", type=int, default=4096)
    training.add_argument(
        "--neural-categorical-transform",
        choices=("embedding", "target"),
        default="embedding",
    )
    training.add_argument(
        "--patience", type=int, help="Defaults: CatBoost 100 iterations; MLP 5 epochs."
    )

    blending = commands.add_parser(
        "blend", help="Average compatible runs with explicit, normalized weights."
    )
    blending.add_argument("--runs", type=Path, nargs="+", required=True)
    blending.add_argument("--weights", type=float, nargs="+")
    blending.add_argument("--name", required=True)
    blending.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    blending.add_argument("--id-column", default="id")
    blending.add_argument("--target", default="is_fraud")

    validation = commands.add_parser(
        "validate-submission",
        help="Require exact schema, ID order and finite probabilities.",
    )
    validation.add_argument("--file", type=Path, required=True)
    validation.add_argument(
        "--sample", type=Path, default=Path("data/raw/sample_submission.csv")
    )
    validation.add_argument("--id-column", default="id")
    validation.add_argument("--target", default="is_fraud")

    holdout = commands.add_parser(
        "evaluate-holdout",
        help="Evaluate frozen development choices once on the reserved independent holdout.",
    )
    holdout.add_argument("--runs", type=Path, nargs="+", required=True)
    holdout.add_argument("--weights", type=float, nargs="+")
    holdout.add_argument(
        "--output", type=Path, default=Path("artifacts/holdout/evaluation.json")
    )
    holdout.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    holdout.add_argument(
        "--development-run",
        type=Path,
        help="The matching pre-holdout development blend, for an explicit provenance chain.",
    )
    final = commands.add_parser(
        "refit",
        help="Refit one frozen selected model on all labeled data, without early stopping.",
    )
    final.add_argument("--run", type=Path, required=True)
    final.add_argument(
        "--holdout-report", type=Path, default=Path("artifacts/holdout/evaluation.json")
    )
    final.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    final.add_argument("--name", required=True)
    public = commands.add_parser(
        "import-public-gpt2",
        help="Validate the approved licensed public GPT-2 arrays and convert logits to the OOF contract.",
    )
    public.add_argument("--source", type=Path, required=True)
    public.add_argument("--data", type=Path, default=Path("data/raw"))
    public.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    public.add_argument("--name", default="explore5-public-gpt2")
    return root


def validate_training_config(config: dict) -> None:
    if config.get("phase") == "exploratory_full_data_cv" and (
        config["holdout_fraction"]
        or config["split"] != "stratified"
        or config["folds"] != 5
        or config["seed"] != 42
    ):
        raise ValueError(
            "Exploratory full-data comparison requires stratified 5-fold/seed42 CV with no independent holdout."
        )
    if config["id_column"] == config["target"]:
        raise ValueError("ID and target must be different columns.")
    if not config["validation_note"].strip():
        raise ValueError("A non-empty validation note is required.")
    if not 0 <= config["holdout_fraction"] < 0.5:
        raise ValueError("holdout_fraction must be in [0, 0.5).")
    if config["holdout_fraction"] and config["split"] != "stratified":
        raise ValueError(
            "The independent random holdout is supported only for stratified IID experiments."
        )
    for name in (
        "threads",
        "iterations",
        "epochs",
        "batch_size",
        "patience",
        "depth",
        "num_leaves",
        "min_child_samples",
    ):
        if config[name] < 1:
            raise ValueError(f"{name} must be positive.")
    if config["num_leaves"] < 2:
        raise ValueError("num_leaves must be at least two.")
    import math

    for name in ("learning_rate", "neural_learning_rate", "logistic_c"):
        if not math.isfinite(config[name]) or config[name] <= 0:
            raise ValueError(f"{name} must be positive.")
    if not 0 <= config["dropout"] < 1 or any(
        width < 1 for width in config["hidden_dims"]
    ):
        raise ValueError(
            "Dropout must be in [0, 1) and hidden layer widths must be positive."
        )
    if (
        not math.isfinite(config["l2_leaf_reg"])
        or not math.isfinite(config["weight_decay"])
        or config["l2_leaf_reg"] < 0
        or config["weight_decay"] < 0
        or not math.isfinite(config["target_smoothing"])
        or config["target_smoothing"] < 0
    ):
        raise ValueError("Regularization values must be nonnegative.")
    if config["split"] != "group" and config["group_column"]:
        raise ValueError("--group-column requires --split group.")
    if config["split"] != "time" and config["time_column"]:
        raise ValueError("--time-column requires --split time.")
    if (
        config["group_column"] in (config["target"], config["id_column"])
        or config["time_column"] == config["target"]
    ):
        raise ValueError(
            "The target cannot define a split; entity grouping must use an actual entity column."
        )


def main() -> None:
    arguments = parser().parse_args()
    try:
        if arguments.command == "inspect":
            profile = inspect_data(
                Path(arguments.data), arguments.id_column, arguments.target
            )
            write_json(Path(arguments.output), profile)
            print(
                json.dumps(
                    {
                        "profile": arguments.output,
                        "rows": profile["rows"],
                        "target": profile["target"],
                        "duplicate_feature_rows": profile["duplicate_feature_rows"],
                        "potential_split_columns": profile["potential_split_columns"],
                    },
                    indent=2,
                )
            )
        elif arguments.command == "train":
            config = vars(arguments).copy()
            config.pop("command")
            if config["patience"] is None:
                config["patience"] = 5 if config["model"] == "mlp" else 100
            validate_training_config(config)
            metrics = train(config)
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in (
                            "model",
                            "roc_auc",
                            "fold_scores",
                            "fold_std",
                            "elapsed_seconds",
                        )
                    },
                    indent=2,
                )
            )
        elif arguments.command == "blend":
            metrics = blend(
                arguments.runs,
                arguments.weights,
                arguments.output,
                arguments.name,
                arguments.id_column,
                arguments.target,
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "roc_auc", "fold_scores", "fold_std")
                    },
                    indent=2,
                )
            )
        elif arguments.command == "evaluate-holdout":
            from .holdout import evaluate

            print(
                json.dumps(
                    evaluate(
                        arguments.runs,
                        arguments.weights,
                        arguments.output,
                        arguments.device,
                        arguments.development_run,
                    ),
                    indent=2,
                )
            )
        elif arguments.command == "refit":
            from .refit import refit

            metrics = refit(
                arguments.run,
                arguments.holdout_report,
                arguments.output,
                arguments.name,
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "phase", "rows", "elapsed_seconds")
                    },
                    indent=2,
                )
            )
        elif arguments.command == "import-public-gpt2":
            from .external import import_public_gpt2

            metrics = import_public_gpt2(
                arguments.source, arguments.data, arguments.output, arguments.name
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "phase", "roc_auc", "fold_scores")
                    },
                    indent=2,
                )
            )
        else:
            frame, sample = pd.read_csv(arguments.file), pd.read_csv(arguments.sample)
            validate_submission(frame, sample, arguments.id_column, arguments.target)
            print(
                json.dumps(
                    {"valid": True, "rows": len(frame), "file": str(arguments.file)}
                )
            )
    except (ValueError, RuntimeError, OSError) as error:
        raise SystemExit(f"Error: {error}") from error


if __name__ == "__main__":
    main()


In [ ]:
%%writefile fraudlab/data.py
"""Input contracts and aggregate inspection. Raw data stays local."""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd


def fingerprint(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def validate_ids(frame: pd.DataFrame, id_column: str) -> None:
    if (
        frame.empty
        or id_column not in frame
        or frame[id_column].isna().any()
        or frame[id_column].duplicated().any()
    ):
        raise ValueError(
            f"{id_column}: rows and IDs must exist; IDs must be non-null and unique."
        )


def validate_submission(
    frame: pd.DataFrame,
    expected: pd.DataFrame,
    id_column: str = "id",
    target: str = "is_fraud",
) -> None:
    if list(frame.columns) != [id_column, target]:
        raise ValueError(
            f"Submission columns must be exactly {[id_column, target]} in that order."
        )
    validate_ids(frame, id_column)
    validate_ids(expected, id_column)
    if frame[id_column].tolist() != expected[id_column].tolist():
        raise ValueError(
            "Submission IDs must match the expected IDs and their order exactly."
        )
    try:
        probabilities = frame[target].to_numpy(dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError("Submission probabilities must be numeric.") from error
    if (
        not np.isfinite(probabilities).all()
        or ((probabilities < 0) | (probabilities > 1)).any()
    ):
        raise ValueError(
            "Submission probabilities must be finite values between 0 and 1."
        )


def load_data(
    directory: Path, id_column: str = "id", target: str = "is_fraud"
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(directory / "train.csv")
    test = pd.read_csv(directory / "test.csv")
    sample = pd.read_csv(directory / "sample_submission.csv")
    if train.empty or test.empty:
        raise ValueError("Training and test data must not be empty.")
    validate_ids(train, id_column)
    validate_ids(test, id_column)
    if set(train[id_column]) & set(test[id_column]):
        raise ValueError("Training and test row IDs must be disjoint.")
    if (
        target not in train
        or train[target].isna().any()
        or set(train[target].unique()) != {0, 1}
    ):
        raise ValueError(
            f"{target}: expected a non-null binary target containing both 0 and 1."
        )
    features = [column for column in train if column not in (id_column, target)]
    if not features or set(test.columns) != {id_column, *features}:
        raise ValueError(
            "Training and test feature schemas must match, excluding target and ID."
        )
    if list(sample.columns) != [id_column, target]:
        raise ValueError("sample_submission.csv has an unexpected schema.")
    validate_ids(sample, id_column)
    if set(sample[id_column]) != set(test[id_column]):
        raise ValueError("Sample submission IDs must cover the test IDs exactly.")
    return train, test[[id_column, *features]], sample


def submission(
    test_ids: pd.Series, predictions: np.ndarray, sample: pd.DataFrame, target: str
) -> pd.DataFrame:
    values = pd.Series(np.asarray(predictions), index=test_ids)
    result = sample.copy()
    result[target] = values.reindex(sample[test_ids.name]).to_numpy()
    validate_submission(result, sample, test_ids.name, target)
    return result


def inspect_data(
    directory: Path, id_column: str = "id", target: str = "is_fraud"
) -> dict:
    train, test, _ = load_data(directory, id_column, target)
    features = [column for column in test if column != id_column]
    columns = {}
    for name in features:
        columns[name] = {
            "dtype": str(train[name].dtype),
            "train_missing": int(train[name].isna().sum()),
            "test_missing": int(test[name].isna().sum()),
            "train_unique": int(train[name].nunique(dropna=False)),
            "test_unique": int(test[name].nunique(dropna=False)),
        }
    duplicate_mask = train.duplicated(features, keep=False)
    return {
        "rows": {"train": len(train), "test": len(test)},
        "target": {
            "name": target,
            "positive_rate": float(train[target].mean()),
            "counts": {str(k): int(v) for k, v in train[target].value_counts().items()},
        },
        "features": columns,
        "duplicate_feature_rows": int(duplicate_mask.sum()),
        "train_test_feature_overlap": int(
            pd.util.hash_pandas_object(train[features], index=False)
            .isin(pd.util.hash_pandas_object(test[features], index=False))
            .sum()
        ),
        "potential_split_columns": [
            name
            for name in features
            if any(
                word in name.lower()
                for word in (
                    "date",
                    "time",
                    "customer",
                    "account",
                    "user",
                    "card_id",
                    "merchant_id",
                )
            )
        ],
        "validation_warning": "Inspect time, entities and duplicate rows before choosing a split. Candidate names alone do not establish the required split.",
        "data_hashes": {
            name: fingerprint(directory / name)
            for name in ("train.csv", "test.csv", "sample_submission.csv")
        },
    }


In [ ]:
%%writefile fraudlab/experiment.py
"""Cross-validation, immutable run artifacts and explicit-weight blending."""

from __future__ import annotations

import importlib.metadata
import json
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from .data import (
    fingerprint,
    load_data,
    submission,
    validate_ids,
    validate_submission,
    write_json,
)
from .features import transform_features
from .models import feature_types, fit_predict
from .splits import make_splits


def versions() -> dict:
    result = {"python": platform.python_version()}
    for package in (
        "fraudlab",
        "numpy",
        "pandas",
        "scikit-learn",
        "catboost",
        "lightgbm",
        "torch",
    ):
        try:
            result[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            continue
    return result


def git_revision() -> str | None:
    result = subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=False
    )
    return result.stdout.strip() if result.returncode == 0 else None


def source_provenance() -> dict:
    status = subprocess.run(
        ["git", "status", "--porcelain"], capture_output=True, text=True, check=False
    )
    return {
        "source_hashes": {
            file.name: fingerprint(file)
            for file in sorted(Path(__file__).parent.glob("*.py"))
        },
        "git_revision": git_revision(),
        "git_dirty": bool(status.stdout.strip()) if status.returncode == 0 else None,
        "source_hashes_timing": "captured before reading data or fitting models",
    }


def create_run(output: Path, name: str) -> Path:
    if not name or Path(name).name != name or name in (".", ".."):
        raise ValueError("Run name must be a single directory name.")
    directory = output / name
    try:
        directory.mkdir(parents=True, exist_ok=False)
    except FileExistsError as error:
        raise FileExistsError(
            f"Run already exists: {directory}. Choose a new run name."
        ) from error
    return directory


def train(config: dict) -> dict:
    started = time.perf_counter()
    provenance = source_provenance()
    data_dir = Path(config["data"])
    frame, test, sample = load_data(data_dir, config["id_column"], config["target"])
    target, id_column = config["target"], config["id_column"]
    total_rows = len(frame)
    holdout_fraction = config.get("holdout_fraction", 0.0)
    if holdout_fraction:
        development, _ = train_test_split(
            np.arange(len(frame)),
            test_size=holdout_fraction,
            stratify=frame[target],
            random_state=config["holdout_seed"],
        )
        frame = frame.iloc[np.sort(development)].reset_index(drop=True)
    frame = transform_features(
        frame, config.get("feature_set", "raw"), (id_column, target)
    )
    test = transform_features(
        test, config.get("feature_set", "raw"), (id_column, target)
    )
    excluded = [id_column, target]
    if config["split"] == "group":
        excluded.append(config["group_column"])
    if config["split"] == "time":
        excluded.append(config["time_column"])
    features = [name for name in frame if name not in excluded]
    if not features:
        raise ValueError(
            "No model features remain after excluding ID, target and split columns."
        )
    splits = make_splits(
        frame,
        target,
        config["split"],
        config["folds"],
        config["seed"],
        config["group_column"],
        config["time_column"],
    )
    run = create_run(Path(config["output"]), config["name"])
    write_json(run / "config.json", config)
    config_hash = fingerprint(run / "config.json")
    write_json(
        run / "status.json",
        {"status": "running", "started_at": datetime.now(timezone.utc).isoformat()},
    )
    hashes = {
        name: fingerprint(data_dir / name)
        for name in ("train.csv", "test.csv", "sample_submission.csv")
    }
    oof = np.full(len(frame), np.nan)
    fold_ids = np.full(len(frame), -1, dtype=int)
    test_predictions = np.zeros(len(test))
    scores, details = [], []
    try:
        for fold, (training_rows, validation_rows) in enumerate(splits):
            valid_prediction, test_prediction, model_details = fit_predict(
                config["model"],
                frame.iloc[training_rows][features],
                frame.iloc[training_rows][target].to_numpy(),
                frame.iloc[validation_rows][features],
                frame.iloc[validation_rows][target].to_numpy(),
                test[features],
                config,
                config["seed"] + fold,
                run / f"fold-{fold}",
            )
            if (
                not np.isfinite(valid_prediction).all()
                or not np.isfinite(test_prediction).all()
            ):
                raise ValueError("Model produced non-finite predictions.")
            oof[validation_rows] = valid_prediction
            fold_ids[validation_rows] = fold
            test_predictions += test_prediction / len(splits)
            score = float(
                roc_auc_score(frame.iloc[validation_rows][target], valid_prediction)
            )
            scores.append(score)
            details.append(model_details)
            print(
                f"fold={fold} roc_auc={score:.8f} train={len(training_rows)} validation={len(validation_rows)}",
                flush=True,
            )
        covered = np.isfinite(oof)
        oof_frame = pd.DataFrame(
            {
                id_column: frame.loc[covered, id_column],
                "fold": fold_ids[covered],
                "target": frame.loc[covered, target],
                "prediction": oof[covered],
            }
        )
        result = submission(test[id_column], test_predictions, sample, target)
        oof_frame.to_csv(run / "oof.csv", index=False)
        result.to_csv(run / "predictions.csv", index=False)
        result.to_csv(run / "submission.csv", index=False)
        numeric, categorical = feature_types(frame[features])
        metrics = {
            "status": "complete",
            "phase": config.get("phase", "development"),
            "config_sha256": config_hash,
            "model": config["model"],
            "roc_auc": float(roc_auc_score(frame.loc[covered, target], oof[covered])),
            "fold_scores": scores,
            "fold_std": float(np.std(scores)),
            "rows": {
                "train": total_rows,
                "test": len(test),
                "oof": int(covered.sum()),
                **(
                    {"development": len(frame), "holdout": total_rows - len(frame)}
                    if holdout_fraction
                    else {}
                ),
            },
            "features": {
                "count": len(features),
                "numeric": len(numeric),
                "categorical": len(categorical),
                "names": features,
                "excluded": list(dict.fromkeys(excluded)),
            },
            "target": {"name": target, "positive_rate": float(frame[target].mean())},
            "seed": config["seed"],
            "validation": {
                "method": config["split"],
                "n_splits": len(splits),
                "note": config["validation_note"],
                "group_column": config["group_column"],
                "time_column": config["time_column"],
                "holdout_fraction": holdout_fraction,
                "holdout_seed": (
                    config.get("holdout_seed") if holdout_fraction else None
                ),
                "independent_holdout": bool(holdout_fraction),
                "early_stopping_uses_validation": config["model"]
                in ("catboost", "lightgbm", "mlp")
                and not config.get("fixed_training"),
            },
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "versions": versions(),
            "data_hashes": hashes,
            **provenance,
            "model_details": {
                **details[0],
                "feature_set": config.get("feature_set", "raw"),
                "folds": details,
            },
            "created_at": datetime.now(timezone.utc).isoformat(),
            "submission_sha256": fingerprint(run / "submission.csv"),
            "limitations": [
                "OOF scores used for model selection are not an independent final assessment.",
                "Public leaderboard and final private leaderboard scores must be obtained from Kaggle.",
                "Temporal OOF, when selected, excludes the initial training-only window.",
                *(
                    [
                        "Exploratory full-data CV uses all labels, including the historical holdout; no independent holdout remains for this stage."
                    ]
                    if config.get("phase") == "exploratory_full_data_cv"
                    else []
                ),
            ],
        }
        write_json(run / "metrics.json", metrics)
        write_json(run / "status.json", {"status": "complete"})
        return metrics
    except Exception as error:
        write_json(run / "status.json", {"status": "failed", "error": str(error)})
        raise


def blend(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    name: str,
    id_column: str = "id",
    target: str = "is_fraud",
) -> dict:
    started = time.perf_counter()
    provenance = source_provenance()
    if len(runs) < 2:
        raise ValueError("Blending requires at least two runs.")
    coefficients = np.asarray(
        weights if weights is not None else [1.0] * len(runs), dtype=float
    )
    if (
        len(coefficients) != len(runs)
        or not np.isfinite(coefficients).all()
        or (coefficients < 0).any()
        or not np.isfinite(coefficients.sum())
        or coefficients.sum() <= 0
    ):
        raise ValueError(
            "Provide one finite nonnegative weight per run, with a positive total."
        )
    coefficients /= coefficients.sum()
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    phases = [metric.get("phase") == "full_data_refit" for metric in metrics]
    if any(phases):
        if not all(phases):
            raise ValueError("Cannot mix development OOF runs with full-data refits.")
        from .refit import blend_refits

        return blend_refits(runs, weights, output, name, id_column, target)
    oofs = [pd.read_csv(run / "oof.csv") for run in runs]
    predictions = [pd.read_csv(run / "predictions.csv") for run in runs]
    reference, expected = oofs[0], predictions[0]
    validate_ids(reference, id_column)
    for index, (score, oof, prediction) in enumerate(zip(metrics, oofs, predictions)):
        if (
            score.get("status") != "complete"
            or score["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Blend sources must be complete runs with identical data fingerprints."
            )
        if not reference[[id_column, "fold", "target"]].equals(
            oof[[id_column, "fold", "target"]]
        ):
            raise ValueError(
                "Blend sources must use exactly the same OOF rows, order, folds and labels."
            )
        if score["validation"] != metrics[0]["validation"]:
            # Models can differ in early stopping; the actual split contract cannot.
            fields = (
                "method",
                "n_splits",
                "group_column",
                "time_column",
                "holdout_fraction",
            )
            if any(
                score["validation"].get(field) != metrics[0]["validation"].get(field)
                for field in fields
            ):
                raise ValueError("Blend sources must use the same validation strategy.")
            if metrics[0]["validation"].get("holdout_fraction", 0) and score[
                "validation"
            ].get("holdout_seed") != metrics[0]["validation"].get("holdout_seed"):
                raise ValueError("Blend sources must reserve exactly the same holdout.")
        validate_submission(prediction, expected, id_column, target)
        if (
            not np.isfinite(oof["prediction"]).all()
            or not oof["prediction"].between(0, 1).all()
        ):
            raise ValueError(f"Source {index} has invalid OOF probabilities.")
    oof_result, prediction_result = reference.copy(), expected.copy()
    oof_result["prediction"] = sum(
        weight * frame["prediction"].to_numpy()
        for weight, frame in zip(coefficients, oofs)
    )
    prediction_result[target] = sum(
        weight * frame[target].to_numpy()
        for weight, frame in zip(coefficients, predictions)
    )
    fold_scores = [
        float(roc_auc_score(group["target"], group["prediction"]))
        for _, group in oof_result.groupby("fold")
    ]
    run = create_run(output, name)
    oof_result.to_csv(run / "oof.csv", index=False)
    prediction_result.to_csv(run / "predictions.csv", index=False)
    prediction_result.to_csv(run / "submission.csv", index=False)
    external_components = {}
    for weight, metric in zip(coefficients, metrics):
        if weight <= 0:
            continue
        for component in metric.get("model_details", {}).get("external_components", []):
            entry = {
                **component,
                "weight": float(weight) * component.get("weight", 1.0),
                "mixture_space": "probability",
            }
            key = entry["predictions_source_url"]
            if key in external_components:
                external_components[key]["weight"] += entry["weight"]
            else:
                external_components[key] = entry
    result = {
        **metrics[0],
        "model": "blend",
        "external_components": list(external_components.values()),
        "roc_auc": float(roc_auc_score(oof_result["target"], oof_result["prediction"])),
        "fold_scores": fold_scores,
        "fold_std": float(np.std(fold_scores)),
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "created_at": datetime.now(timezone.utc).isoformat(),
        **provenance,
        "model_details": {
            "method": "weighted_probability_average",
            "sources": [str(source) for source in runs],
            "weights": coefficients.tolist(),
            "external_components": list(external_components.values()),
            "mixture_space": "probability",
        },
        "submission_sha256": fingerprint(run / "submission.csv"),
    }
    write_json(
        run / "config.json",
        {"sources": [str(source) for source in runs], "weights": coefficients.tolist()},
    )
    result["config_sha256"] = fingerprint(run / "config.json")
    write_json(run / "metrics.json", result)
    write_json(run / "status.json", {"status": "complete"})
    return result


In [ ]:
%%writefile fraudlab/external.py
"""Import one explicitly approved public prediction release; never execute its code."""

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from .data import fingerprint, load_data, submission, write_json
from .experiment import create_run, source_provenance, versions
from .models import feature_types
from .splits import make_splits

MANIFEST_SHA256 = "0083d23b46f71311ceeed66874b093ec927d25d98b37f6d2e78b1de17f0f22d0"
DATASET = "https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof"
CODE = "https://github.com/Matapanino/s6e11-gpt2-target-first"
CODE_REVISION = "8446fb73c11583df4f7006aa08fa34472249b318"


def import_public_gpt2(source: Path, data: Path, output: Path, name: str) -> dict:
    provenance = source_provenance()
    if fingerprint(source / "manifest.json") != MANIFEST_SHA256:
        raise ValueError(
            "Public manifest does not match the reviewed, approved release."
        )
    manifest = json.loads((source / "manifest.json").read_text(encoding="utf-8"))
    hashes = {
        filename: fingerprint(data / filename)
        for filename in ("train.csv", "test.csv", "sample_submission.csv")
    }
    if (
        hashes["train.csv"] != manifest["inputs"]["train_csv"]["sha256"]
        or hashes["test.csv"] != manifest["inputs"]["test_csv"]["sha256"]
    ):
        raise ValueError(
            "Public arrays are not aligned with these exact train/test CSV bytes."
        )
    expected = {
        **manifest["sha256"],
        "fold_ids.npy": manifest["inputs"]["fold_ids"]["sha256"],
    }
    arrays = {}
    for filename in ("oof.npy", "test.npy", "fold_ids.npy"):
        if fingerprint(source / filename) != expected[filename]:
            raise ValueError(f"Public array hash mismatch: {filename}")
        arrays[filename] = np.load(source / filename, allow_pickle=False)
    train, test, sample = load_data(data)
    for filename, rows in (("oof.npy", len(train)), ("test.npy", len(test))):
        array = arrays[filename]
        if (
            array.shape != (rows,)
            or array.dtype != np.float64
            or not np.isfinite(array).all()
        ):
            raise ValueError(
                f"Expected finite float64 logits in original CSV row order: {filename}"
            )
    fold_ids = arrays["fold_ids.npy"]
    if fold_ids.shape != (len(train),) or fold_ids.dtype != np.int8:
        raise ValueError("Expected int8 fold IDs, one per original training row.")
    expected_folds = np.full(len(train), -1, dtype=np.int8)
    for fold, (_, validation) in enumerate(
        make_splits(train, "is_fraud", "stratified", 5, 42)
    ):
        expected_folds[validation] = fold
    if not np.array_equal(fold_ids, expected_folds):
        raise ValueError(
            "Public folds differ from exact StratifiedKFold(5, shuffle=True, seed42)."
        )
    # Stable sigmoid: the approved arrays contain log-odds, not probabilities.
    oof_probability = np.exp(-np.logaddexp(0.0, -arrays["oof.npy"]))
    test_probability = np.exp(-np.logaddexp(0.0, -arrays["test.npy"]))
    score = float(roc_auc_score(train["is_fraud"], oof_probability))
    if abs(score - manifest["auc"]["pooled"]) > 1e-10:
        raise ValueError(
            "Reproduced public OOF AUC differs from the approved manifest."
        )
    fold_scores = [
        float(
            roc_auc_score(
                train.loc[fold_ids == fold, "is_fraud"],
                oof_probability[fold_ids == fold],
            )
        )
        for fold in range(5)
    ]
    result = submission(test["id"], test_probability, sample, "is_fraud")
    run = create_run(output, name)
    pd.DataFrame(
        {
            "id": train["id"],
            "fold": fold_ids,
            "target": train["is_fraud"],
            "prediction": oof_probability,
        }
    ).to_csv(run / "oof.csv", index=False)
    result.to_csv(run / "predictions.csv", index=False)
    result.to_csv(run / "submission.csv", index=False)
    raw_features = train.drop(columns=["id", "is_fraud"])
    numeric, categorical = feature_types(raw_features)
    component = {
        "name": "GPT-2 target-first",
        "kind": "gpt2_target_first",
        "source_type": "external_predictions",
        "trained_by": "Masaya Kawamata",
        "model_architecture": "GPT-2 target-first classifier",
        "training_code_url": CODE,
        "training_code_commit": CODE_REVISION,
        "code_license": "MIT",
        "predictions_source_url": DATASET,
        "predictions_source_version": 2,
        "predictions_download_ref": "masayakawamata/s6e11-gpt2-target-first-oof/2",
        "predictions_license": "CC BY 4.0",
        "data_license": "CC BY 4.0 (prediction dataset)",
        "method_credits": [
            "Chris Deotte: target-first GPT-2 method",
            "Borisov et al.: GReaT / Language Models are Realistic Tabular Data Generators",
            "uditjain13: original Credit Card Fraud Detection 2026 dataset",
        ],
        "training_executed_locally": False,
        "trained_by_us": False,
        "oof_auc": score,
        "validation": {
            "method": "StratifiedKFold",
            "n_splits": 5,
            "shuffle": True,
            "seed": 42,
        },
        "inference": {
            "method": "target-first label likelihood ratio",
            "target_first": True,
            "likelihood_ratio": True,
            "prior": manifest["settings"]["prior"],
            "orders": manifest["settings"]["orders"],
            "order_aggregation": manifest["settings"]["combine"],
            "test_aggregation": "mean of fold log-odds",
            "test_ensemble_folds": 5,
            "source_test_aggregation": manifest["settings"]["test"],
            "import_transform": "sigmoid(log_odds), float64",
        },
    }
    metrics = {
        "status": "complete",
        "model": "external_gpt2",
        "external_components": [component],
        "phase": "exploratory_full_data_cv",
        "roc_auc": score,
        "fold_scores": fold_scores,
        "fold_std": float(np.std(fold_scores)),
        "rows": {"train": len(train), "test": len(test), "oof": len(train)},
        "features": {
            "count": raw_features.shape[1],
            "numeric": len(numeric),
            "categorical": len(categorical),
            "names": raw_features.columns.tolist(),
            "excluded": ["id", "is_fraud"],
        },
        "target": {
            "name": "is_fraud",
            "positive_rate": float(train["is_fraud"].mean()),
        },
        "seed": 42,
        "validation": {
            "method": "stratified",
            "n_splits": 5,
            "note": "Exploratory full-data CV with externally trained predictions; all labels, including the historical holdout, were used in fold training. The old holdout is not independent for this stage.",
            "group_column": None,
            "time_column": None,
            "holdout_fraction": 0.0,
            "holdout_seed": None,
            "independent_holdout": False,
            "early_stopping_uses_validation": False,
        },
        "model_details": {
            "external_components": [component],
            "training_executed_locally": False,
        },
        "external_provenance": {
            "manifest_sha256": MANIFEST_SHA256,
            "array_sha256": expected,
            "fold_ids_reproduced": True,
            "row_alignment": "Exact raw train/test SHA256 and original row order; no sorting by ID",
        },
        "data_hashes": hashes,
        "versions": versions(),
        **provenance,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "submission_sha256": fingerprint(run / "submission.csv"),
        "limitations": [
            "This neural component was trained by an external author, not by FraudLab. Preserve CC BY 4.0 attribution.",
            "No independent holdout is available for this exploratory full-data stage; the earlier holdout report applies only to the original protocol.",
            "Public test predictions average fold log-odds before sigmoid; OOF rows use only their excluded-fold model.",
        ],
    }
    write_json(
        run / "config.json",
        {
            "model": "external_gpt2",
            "phase": "exploratory_full_data_cv",
            "source": DATASET,
            "manifest_sha256": MANIFEST_SHA256,
            "logits_to_probabilities": "sigmoid",
        },
    )
    write_json(run / "metrics.json", metrics)
    write_json(run / "status.json", {"status": "complete"})
    return metrics


In [ ]:
%%writefile fraudlab/features.py
"""Small deterministic hypotheses; every feature set is compared by ablation."""

from __future__ import annotations

import numpy as np
import pandas as pd

CONTINUOUS = (
    "amount_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "account_balance_usd",
    "velocity_score",
    "merchant_risk_score",
)


def transform_features(
    frame: pd.DataFrame,
    feature_set: str = "raw",
    excluded: tuple[str, ...] = ("id", "is_fraud"),
) -> pd.DataFrame:
    if feature_set == "raw":
        return frame.copy()
    if feature_set not in ("behavior", "digits", "value_digits"):
        raise ValueError(f"Unknown feature set: {feature_set}")
    result = frame.copy()
    for name in CONTINUOUS:
        if name in result and name not in excluded:
            values = result[name].to_numpy(dtype=float)
            result[name + "__log1p"] = np.sign(values) * np.log1p(np.abs(values))
    for name, period in (("time_of_day_hour", 24), ("day_of_week", 7)):
        if name in result and name not in excluded:
            angle = 2 * np.pi * result[name] / period
            result[name + "__sin"] = np.sin(angle)
            result[name + "__cos"] = np.cos(angle)
    if {"amount_usd", "account_balance_usd"}.issubset(result) and not {
        "amount_usd",
        "account_balance_usd",
    }.intersection(excluded):
        result["amount_balance_ratio"] = result["amount_usd"] / (
            np.abs(result["account_balance_usd"]) + 1
        )
    if feature_set in ("digits", "value_digits"):
        # These views test synthetic rounding patterns; they are not banking rules.
        for name in CONTINUOUS:
            if name not in result or name in excluded:
                continue
            values = np.abs(result[name].to_numpy(dtype=float))
            if not np.isfinite(values).all():
                raise ValueError("Digit views require finite numeric inputs.")
            cents = np.rint(values * 100).astype(np.int64)
            magnitude = np.floor(np.log10(np.maximum(values, 1e-12))).astype(int)
            leading = np.floor(values / np.power(10.0, magnitude)).astype(int)
            result[name + "__value_key"] = pd.Series(cents, index=result.index).astype(
                str
            )
            result[name + "__cents_key"] = pd.Series(
                cents % 100, index=result.index
            ).astype(str)
            result[name + "__leading_key"] = pd.Series(
                leading, index=result.index
            ).astype(str)
            if feature_set == "value_digits":
                leading_two = np.floor(values / np.power(10.0, magnitude - 1)).astype(
                    int
                )
                result[name + "__integer_key"] = pd.Series(
                    cents // 100, index=result.index
                ).astype(str)
                result[name + "__tenths_key"] = pd.Series(
                    cents // 10, index=result.index
                ).astype(str)
                result[name + "__leading_two_key"] = pd.Series(
                    leading_two, index=result.index
                ).astype(str)
        if feature_set == "value_digits":
            for name in frame.select_dtypes(include="number"):
                if name not in CONTINUOUS and name not in excluded:
                    result[name + "__value_key"] = frame[name].astype(str)
    return result


In [ ]:
%%writefile fraudlab/holdout.py
"""One final evaluation after development choices have been frozen."""

from __future__ import annotations

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from .data import fingerprint, load_data, write_json
from .features import transform_features
from .models import predict_saved


def evaluate(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    device: str = "auto",
    development_run: Path | None = None,
) -> dict:
    if output.exists():
        raise FileExistsError(
            "Holdout evaluation already exists; do not reuse it for model selection."
        )
    if not runs:
        raise ValueError("At least one frozen source run is required.")
    coefficients = np.asarray(
        weights if weights is not None else [1.0] * len(runs), dtype=float
    )
    if (
        len(coefficients) != len(runs)
        or not np.isfinite(coefficients).all()
        or not np.isfinite(coefficients.sum())
        or (coefficients < 0).any()
        or coefficients.sum() <= 0
    ):
        raise ValueError(
            "Provide finite, nonnegative weights with a finite positive total."
        )
    coefficients /= coefficients.sum()
    configs = [
        json.loads((run / "config.json").read_text(encoding="utf-8")) for run in runs
    ]
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    reference = configs[0]
    if not reference.get("holdout_fraction"):
        raise ValueError("Sources must have reserved a holdout before training.")
    fields = ("data", "target", "id_column", "holdout_fraction", "holdout_seed")
    for run, config, metric in zip(runs, configs, metrics):
        if (
            metric.get("config_sha256")
            and fingerprint(run / "config.json") != metric["config_sha256"]
        ):
            raise ValueError("Source configuration changed after training.")
        if config["model"] != metric["model"] or config.get(
            "feature_set", "raw"
        ) != metric["model_details"].get("feature_set", "raw"):
            raise ValueError(
                "Source model or feature set no longer matches its training metadata."
            )
        if any(
            config.get(field) != metric["validation"].get(field)
            for field in ("holdout_fraction", "holdout_seed")
        ):
            raise ValueError(
                "Source config no longer matches the holdout recorded at training time."
            )
        if (
            any(config.get(field) != reference.get(field) for field in fields)
            or metric.get("status") != "complete"
            or metric["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Sources must be complete runs sharing the same untouched holdout and input data."
            )
    data_dir = Path(reference["data"])
    for filename, expected in metrics[0]["data_hashes"].items():
        if fingerprint(data_dir / filename) != expected:
            raise ValueError("Input files changed after training.")
    frame, _, _ = load_data(data_dir, reference["id_column"], reference["target"])
    _, held_out = train_test_split(
        np.arange(len(frame)),
        test_size=reference["holdout_fraction"],
        stratify=frame[reference["target"]],
        random_state=reference["holdout_seed"],
    )
    frame = frame.iloc[np.sort(held_out)].reset_index(drop=True)
    held_out_ids = set(frame[reference["id_column"]])
    for run in runs:
        if held_out_ids.intersection(
            pd.read_csv(run / "oof.csv", usecols=[reference["id_column"]])[
                reference["id_column"]
            ]
        ):
            raise ValueError("Holdout rows were found in development OOF predictions.")
    output.parent.mkdir(parents=True, exist_ok=True)
    # Persist the chosen configuration before the first holdout prediction is evaluated.
    selection = {
        "sources": [str(run) for run in runs],
        "weights": coefficients.tolist(),
        "metrics_hashes": [fingerprint(run / "metrics.json") for run in runs],
        "config_hashes": [fingerprint(run / "config.json") for run in runs],
        "frozen_at": datetime.now(timezone.utc).isoformat(),
    }
    if development_run is not None:
        dev_config = json.loads(
            (development_run / "config.json").read_text(encoding="utf-8")
        )
        dev_metric = json.loads(
            (development_run / "metrics.json").read_text(encoding="utf-8")
        )
        if (
            [Path(source).resolve() for source in dev_config["sources"]]
            != [source.resolve() for source in runs]
            or not np.allclose(dev_config["weights"], coefficients, rtol=0, atol=1e-12)
            or dev_metric.get("status") != "complete"
            or dev_metric["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Development blend does not match the frozen sources, weights and input data."
            )
        selection.update(
            source_dev_run=str(development_run),
            development_roc_auc=dev_metric["roc_auc"],
            development_metrics_sha256=fingerprint(development_run / "metrics.json"),
        )
    partition_key = hashlib.sha256(
        json.dumps(
            {
                "train_sha256": metrics[0]["data_hashes"]["train.csv"],
                "target": reference["target"],
                "seed": reference["holdout_seed"],
                "fraction": reference["holdout_fraction"],
            },
            sort_keys=True,
        ).encode()
    ).hexdigest()
    ledger_dir = data_dir / ".holdout-evaluations"
    ledger_dir.mkdir(exist_ok=True)
    ledger = ledger_dir / f"{partition_key}.json"
    selection_key = hashlib.sha256(
        json.dumps(
            {key: value for key, value in selection.items() if key != "frozen_at"},
            sort_keys=True,
        ).encode()
    ).hexdigest()
    if ledger.exists():
        prior = json.loads(ledger.read_text(encoding="utf-8"))
        if prior["status"] == "complete" or prior["selection_key"] != selection_key:
            raise ValueError(
                "This independent holdout partition is already reserved or evaluated; do not retune from it."
            )
    else:
        # The dataset-local ledger follows this partition even if --output changes.
        # A failed evaluation may resume only with the identical frozen sources.
        with ledger.open("x", encoding="utf-8") as stream:
            json.dump(
                {
                    "status": "started",
                    "selection_key": selection_key,
                    "selection": selection,
                },
                stream,
                indent=2,
            )
    write_json(output.with_name(output.stem + "-frozen-selection.json"), selection)
    predictions = np.zeros(len(frame))
    individual = []
    for weight, run, config, metric in zip(coefficients, runs, configs, metrics):
        model_predictions = np.zeros(len(frame))
        inputs = transform_features(
            frame,
            config.get("feature_set", "raw"),
            (config["id_column"], config["target"]),
        )[metric["features"]["names"]]
        for fold in range(metric["validation"]["n_splits"]):
            model_predictions += (
                predict_saved(config["model"], run / f"fold-{fold}", inputs, device)
                / metric["validation"]["n_splits"]
            )
        predictions += weight * model_predictions
        individual.append(
            float(roc_auc_score(frame[reference["target"]], model_predictions))
        )
    if (
        not np.isfinite(predictions).all()
        or ((predictions < 0) | (predictions > 1)).any()
    ):
        raise ValueError("Invalid holdout predictions.")
    result = {
        "status": "complete",
        "kind": "independent_holdout",
        "evaluation_source_hashes": {
            file.name: fingerprint(file)
            for file in sorted(Path(__file__).parent.glob("*.py"))
        },
        "roc_auc": float(roc_auc_score(frame[reference["target"]], predictions)),
        "individual_roc_auc": individual,
        "rows": len(frame),
        "positive_rows": int(frame[reference["target"]].sum()),
        "holdout_seed": reference["holdout_seed"],
        "holdout_fraction": reference["holdout_fraction"],
        "selection": selection,
        "data_hashes": metrics[0]["data_hashes"],
        "evaluated_at": datetime.now(timezone.utc).isoformat(),
        "limitation": "The selection was frozen before this evaluation. Using this result to retune would make the holdout part of model selection.",
    }
    pd.DataFrame(
        {
            reference["id_column"]: frame[reference["id_column"]],
            "target": frame[reference["target"]],
            "prediction": predictions,
        }
    ).to_csv(output.with_suffix(".csv"), index=False)
    write_json(output, result)
    write_json(
        ledger,
        {
            "status": "complete",
            "selection_key": selection_key,
            "report_sha256": fingerprint(output),
            "evaluated_at": result["evaluated_at"],
        },
    )
    return result


In [ ]:
%%writefile fraudlab/models.py
"""Small tabular models, each fitted inside its training fold."""

from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    QuantileTransformer,
    StandardScaler,
    TargetEncoder,
)
from threadpoolctl import threadpool_limits


def feature_types(frame: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric = frame.select_dtypes(include="number").columns.tolist()
    return numeric, [name for name in frame if name not in numeric]


def clean_features(
    frame: pd.DataFrame,
    numeric: list[str] | None = None,
    categorical: list[str] | None = None,
) -> pd.DataFrame:
    result = frame.copy()
    if numeric is None or categorical is None:
        numeric, categorical = feature_types(result)
    for name in numeric:
        result[name] = pd.to_numeric(result[name], errors="raise")
    result[numeric] = result[numeric].replace([np.inf, -np.inf], np.nan)
    for name in categorical:
        result[name] = result[name].astype("string").fillna("__MISSING__").astype(str)
    return result


def log_odds(values: np.ndarray) -> np.ndarray:
    probabilities = np.clip(values, 1e-6, 1 - 1e-6)
    return np.log(probabilities / (1 - probabilities))


def fit_logistic(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    numeric, categorical = feature_types(training)
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        (
                            "impute",
                            SimpleImputer(strategy="median", keep_empty_features=True),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore", min_frequency=5, dtype=np.float32
                ),
                categorical,
            ),
        ],
        sparse_threshold=1.0,
    )
    model = Pipeline(
        [
            ("preprocessing", preprocessing),
            (
                "classifier",
                LogisticRegression(
                    C=config["logistic_c"], max_iter=1000, random_state=seed
                ),
            ),
        ]
    )
    with threadpool_limits(limits=config["threads"]):
        model.fit(training, y)
    joblib.dump(model, path.with_suffix(".joblib"))
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        {
            "algorithm": "logistic_regression",
            "C": config["logistic_c"],
            "solver": "lbfgs",
            "max_iter": 1000,
            "iterations_used": int(model.named_steps["classifier"].n_iter_[0]),
            "preprocessing": "Fold-local median imputation, standard scaling, one-hot encoding; unknown categories ignored; category min_frequency=5.",
        },
    )


def fit_catboost(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    from catboost import CatBoostClassifier

    _, categorical = feature_types(training)
    device = "GPU" if config["device"] == "cuda" else "CPU"
    parameters = {
        "iterations": config["iterations"],
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "l2_leaf_reg": config["l2_leaf_reg"],
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": seed,
        "thread_count": config["threads"],
        "task_type": device,
        "allow_writing_files": False,
        "verbose": False,
    }
    model = CatBoostClassifier(**parameters)
    if config.get("refit") or config.get("fixed_training"):
        model.fit(training, y, cat_features=categorical)
    else:
        model.fit(
            training,
            y,
            cat_features=categorical,
            eval_set=(validation, validation_y),
            early_stopping_rounds=config["patience"],
            use_best_model=True,
        )
    model.save_model(str(path.with_suffix(".cbm")))
    details = {
        **parameters,
        "best_iteration": int(model.get_best_iteration()),
        "trees_used": int(model.tree_count_),
        "early_stopping_patience": (
            None
            if config.get("refit") or config.get("fixed_training")
            else config["patience"]
        ),
        "preprocessing": "Numeric missing values handled by CatBoost; categorical missing values mapped to __MISSING__; native ordered categorical statistics.",
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


class NeuralPreprocessor:
    """Training-fold-only numeric statistics and categorical vocabularies."""

    def __init__(
        self,
        numeric_transform: str = "standard",
        seed: int = 42,
        categorical_transform: str = "embedding",
        smoothing: float = 20.0,
    ):
        self.numeric_transform = numeric_transform
        self.seed = seed
        self.categorical_transform = categorical_transform
        self.smoothing = smoothing

    def fit(self, frame: pd.DataFrame) -> "NeuralPreprocessor":
        self.numeric, self.categorical = feature_types(frame)
        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)
        self.scaler = (
            QuantileTransformer(
                n_quantiles=min(1000, len(frame)),
                output_distribution="normal",
                subsample=min(100000, len(frame)),
                random_state=self.seed,
            )
            if self.numeric_transform == "quantile"
            else StandardScaler()
        )
        if self.numeric:
            self.scaler.fit(self.imputer.fit_transform(frame[self.numeric]))
        self.vocabularies = {
            name: {
                value: index + 1
                for index, value in enumerate(sorted(frame[name].unique()))
            }
            for name in self.categorical
        }
        self.cardinalities = [
            len(self.vocabularies[name]) + 1 for name in self.categorical
        ]
        return self

    def fit_transform(
        self, frame: pd.DataFrame, y: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray]:
        self.fit(frame)
        if self.categorical_transform == "embedding":
            return self.transform(frame)
        numbers = (
            self.scaler.transform(self.imputer.transform(frame[self.numeric])).astype(
                np.float32
            )
            if self.numeric
            else np.empty((len(frame), 0), dtype=np.float32)
        )
        self.cardinalities = []
        self.target_encoder = TargetEncoder(
            target_type="binary",
            smooth=self.smoothing,
            cv=5,
            shuffle=True,
            random_state=self.seed,
        )
        self.target_scaler = StandardScaler()
        if self.categorical:
            encoded = self.target_encoder.fit_transform(frame[self.categorical], y)
            encoded = self.target_scaler.fit_transform(log_odds(encoded)).astype(
                np.float32
            )
            numbers = np.column_stack([numbers, encoded])
        return numbers, np.empty((len(frame), 0), dtype=np.int64)

    def transform(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        numbers = (
            self.scaler.transform(self.imputer.transform(frame[self.numeric])).astype(
                np.float32
            )
            if self.numeric
            else np.empty((len(frame), 0), dtype=np.float32)
        )
        if getattr(self, "categorical_transform", "embedding") == "target":
            if self.categorical:
                encoded = self.target_encoder.transform(frame[self.categorical])
                encoded = self.target_scaler.transform(log_odds(encoded)).astype(
                    np.float32
                )
                numbers = np.column_stack([numbers, encoded])
            return numbers, np.empty((len(frame), 0), dtype=np.int64)
        categories = (
            np.column_stack(
                [
                    frame[name]
                    .map(self.vocabularies[name])
                    .fillna(0)
                    .to_numpy(dtype=np.int64)
                    for name in self.categorical
                ]
            )
            if self.categorical
            else np.empty((len(frame), 0), dtype=np.int64)
        )
        return numbers, categories


def build_network(
    torch,
    numeric_count: int,
    cardinalities: list[int],
    hidden_dims: list[int],
    dropout: float,
):
    nn = torch.nn
    embedding_dims = [
        min(32, max(2, int(np.sqrt(count)) + 1)) for count in cardinalities
    ]

    class TabularMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.embeddings = nn.ModuleList(
                [
                    nn.Embedding(count, width, padding_idx=0)
                    for count, width in zip(cardinalities, embedding_dims)
                ]
            )
            width = numeric_count + sum(embedding_dims)
            layers = []
            for next_width in hidden_dims:
                layers.extend(
                    [nn.Linear(width, next_width), nn.ReLU(), nn.Dropout(dropout)]
                )
                width = next_width
            layers.append(nn.Linear(width, 1))
            self.layers = nn.Sequential(*layers)

        def forward(self, numbers, categories):
            pieces = [
                numbers,
                *[
                    embedding(categories[:, i])
                    for i, embedding in enumerate(self.embeddings)
                ],
            ]
            return self.layers(torch.cat(pieces, dim=1)).squeeze(1)

    return TabularMLP(), embedding_dims


def fit_neural(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    try:
        import torch
    except ImportError as error:
        raise RuntimeError(
            "Neural training requires PyTorch: install fraudlab[neural]."
        ) from error

    torch.set_num_threads(config["threads"])
    torch.manual_seed(seed)
    if config["device"] == "cuda" and not torch.cuda.is_available():
        raise ValueError(
            "CUDA was requested but is not available in this PyTorch installation."
        )
    device = torch.device(
        "cuda" if config["device"] != "cpu" and torch.cuda.is_available() else "cpu"
    )
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    preprocessing = NeuralPreprocessor(
        config.get("neural_numeric_transform", "standard"),
        seed,
        config.get("neural_categorical_transform", "embedding"),
        config.get("target_smoothing", 20.0),
    )
    arrays = [
        preprocessing.fit_transform(training, y),
        preprocessing.transform(validation),
        preprocessing.transform(test),
    ]
    tensors = [
        tuple(torch.as_tensor(value, device=device) for value in pair)
        for pair in arrays
    ]
    labels = torch.as_tensor(y, dtype=torch.float32, device=device)
    model, embedding_dims = build_network(
        torch,
        arrays[0][0].shape[1],
        preprocessing.cardinalities,
        config["hidden_dims"],
        config["dropout"],
    )
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["neural_learning_rate"],
        weight_decay=config["weight_decay"],
    )
    criterion = torch.nn.BCEWithLogitsLoss()

    def predict(pair) -> np.ndarray:
        model.eval()
        with torch.no_grad():
            return np.concatenate(
                [
                    torch.sigmoid(
                        model(
                            pair[0][start : start + config["batch_size"]],
                            pair[1][start : start + config["batch_size"]],
                        )
                    )
                    .cpu()
                    .numpy()
                    for start in range(0, len(pair[0]), config["batch_size"])
                ]
            )

    best_auc, stale, best_epoch, best_state = -np.inf, 0, 0, None
    epochs_completed = 0
    for epoch in range(config["epochs"]):
        model.train()
        order = torch.randperm(len(y), device=device)
        for start in range(0, len(y), config["batch_size"]):
            selected = order[start : start + config["batch_size"]]
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(
                model(tensors[0][0][selected], tensors[0][1][selected]),
                labels[selected],
            )
            if not torch.isfinite(loss):
                raise ValueError(
                    "Neural loss became non-finite; check features or lower the learning rate."
                )
            loss.backward()
            optimizer.step()
        epochs_completed = epoch + 1
        if config.get("refit") or config.get("fixed_training"):
            best_epoch = epochs_completed
            if epochs_completed == config["epochs"]:
                best_state = {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in model.state_dict().items()
                }
            continue
        score = roc_auc_score(validation_y, predict(tensors[1]))
        if score > best_auc + 1e-6:
            best_auc, best_epoch, stale = score, epoch + 1, 0
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
        else:
            stale += 1
        if stale >= config["patience"]:
            break
    if best_state is None:
        raise ValueError("Neural training did not produce a usable checkpoint.")
    model.load_state_dict(best_state)
    details = {
        "architecture": {
            "numeric_features": int(arrays[0][0].shape[1]),
            "raw_numeric_features": len(preprocessing.numeric),
            "categorical_transform": config.get(
                "neural_categorical_transform", "embedding"
            ),
            "categorical_cardinalities": preprocessing.cardinalities,
            "embedding_dims": embedding_dims,
            "hidden_dims": config["hidden_dims"],
            "dropout": config["dropout"],
            "hidden_activation": "ReLU",
            "normalization": "none",
            "output_activation": "sigmoid",
        },
        "optimizer": "AdamW",
        "loss": "BCEWithLogitsLoss",
        "learning_rate": config["neural_learning_rate"],
        "weight_decay": config["weight_decay"],
        "batch_size": config["batch_size"],
        "epochs_completed": epochs_completed,
        "best_epoch": best_epoch,
        "early_stopping_patience": (
            None
            if config.get("refit") or config.get("fixed_training")
            else config["patience"]
        ),
        "device": str(device),
        "numeric_transform": config.get("neural_numeric_transform", "standard"),
        "target_encoder": (
            {
                "internal_folds": 5,
                "smooth": config.get("target_smoothing", 20.0),
                "unknown": "training prior",
                "log_odds": True,
            }
            if config.get("neural_categorical_transform") == "target"
            else None
        ),
        "preprocessing": "Fold-local numeric median imputation and "
        + (
            "quantile-to-normal transformation"
            if config.get("neural_numeric_transform") == "quantile"
            else "standard scaling"
        )
        + (
            "; categorical target-encoding logits cross-fitted into five inner partitions, then scaled using training-only statistics."
            if config.get("neural_categorical_transform") == "target"
            else "; categorical training vocabularies with index 0 reserved for unknown categories."
        ),
    }
    torch.save(
        {"state_dict": best_state, "architecture": details["architecture"]},
        path.with_suffix(".pt"),
    )
    joblib.dump(preprocessing, path.with_suffix(".preprocessing.joblib"))
    return predict(tensors[1]), predict(tensors[2]), details


def fit_predict(
    name: str,
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    numeric, categorical = feature_types(training)
    training, validation, test = [
        clean_features(frame, numeric, categorical)
        for frame in (training, validation, test)
    ]
    if name == "logistic":
        return fit_logistic(training, y, validation, test, config, seed, path)
    if name == "glm":
        return fit_glm(training, y, validation, test, config, seed, path)
    if name == "catboost":
        return fit_catboost(
            training, y, validation, validation_y, test, config, seed, path
        )
    if name == "mlp":
        return fit_neural(
            training, y, validation, validation_y, test, config, seed, path
        )
    if name == "lightgbm":
        return fit_lightgbm(
            training, y, validation, validation_y, test, config, seed, path
        )
    raise ValueError(f"Unknown model: {name}")


def predict_saved(
    name: str, path: Path, frame: pd.DataFrame, device: str = "auto"
) -> np.ndarray:
    """Load only locally trained checkpoints; never download executable artifacts."""
    if name in ("logistic", "glm", "lightgbm"):
        return joblib.load(path.with_suffix(".joblib")).predict_proba(
            clean_features(frame)
        )[:, 1]
    if name == "catboost":
        from catboost import CatBoostClassifier

        model = CatBoostClassifier()
        model.load_model(str(path.with_suffix(".cbm")))
        return model.predict_proba(clean_features(frame))[:, 1]
    if name != "mlp":
        raise ValueError(f"Unknown saved model: {name}")
    import torch

    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    selected_device = torch.device(
        "cuda" if device != "cpu" and torch.cuda.is_available() else "cpu"
    )
    preprocessing = joblib.load(path.with_suffix(".preprocessing.joblib"))
    checkpoint = torch.load(
        path.with_suffix(".pt"), map_location="cpu", weights_only=True
    )
    architecture = checkpoint["architecture"]
    model, _ = build_network(
        torch,
        architecture["numeric_features"],
        architecture["categorical_cardinalities"],
        architecture["hidden_dims"],
        architecture["dropout"],
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.to(selected_device).eval()
    prepared = preprocessing.transform(
        clean_features(frame, preprocessing.numeric, preprocessing.categorical)
    )
    predictions = []
    with torch.no_grad():
        for start in range(0, len(frame), 8192):
            tensors = [
                torch.as_tensor(values[start : start + 8192], device=selected_device)
                for values in prepared
            ]
            predictions.append(torch.sigmoid(model(*tensors)).cpu().numpy())
    return np.concatenate(predictions)


def fit_lightgbm(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    from lightgbm import LGBMClassifier, early_stopping

    numeric, categorical = feature_types(training)
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                SimpleImputer(strategy="median", keep_empty_features=True),
                numeric,
            ),
            (
                "categorical",
                TargetEncoder(
                    target_type="binary",
                    smooth=config.get("target_smoothing", "auto"),
                    cv=5,
                    shuffle=True,
                    random_state=seed,
                ),
                categorical,
            ),
        ]
    )
    # fit_transform cross-fits encodings inside this outer training fold. Validation
    # and test use mappings learned only from that training fold.
    encoded_training = preprocessing.fit_transform(training, y)
    encoded_validation = preprocessing.transform(validation)
    parameters = {
        "n_estimators": config["iterations"],
        "num_leaves": config["num_leaves"],
        "learning_rate": config["learning_rate"],
        "min_child_samples": config["min_child_samples"],
        "reg_lambda": config["l2_leaf_reg"],
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "random_state": seed,
        "n_jobs": config["threads"],
        "deterministic": True,
        "force_col_wise": True,
    }
    classifier = LGBMClassifier(**parameters)
    if config.get("refit") or config.get("fixed_training"):
        classifier.fit(encoded_training, y)
    else:
        classifier.fit(
            encoded_training,
            y,
            eval_X=encoded_validation,
            eval_y=validation_y,
            eval_metric="auc",
            callbacks=[
                early_stopping(
                    config["patience"], first_metric_only=True, verbose=False
                )
            ],
        )
    model = Pipeline([("preprocessing", preprocessing), ("classifier", classifier)])
    joblib.dump(model, path.with_suffix(".joblib"))
    details = {
        **parameters,
        "best_iteration": int(classifier.best_iteration_ or classifier.n_estimators_),
        "target_encoder": {
            "internal_folds": 5,
            "smooth": config.get("target_smoothing", "auto"),
            "unknown": "training prior",
        },
        "preprocessing": "Fold-local median imputation and supervised categorical target encoding, internally cross-fitted into five partitions.",
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


def fit_glm(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    numeric, categorical = feature_types(training)
    nominal = [name for name in categorical if "__" not in name]
    encoding = Pipeline(
        [
            (
                "target",
                TargetEncoder(
                    target_type="binary",
                    smooth=config["target_smoothing"],
                    cv=5,
                    shuffle=True,
                    random_state=seed,
                ),
            ),
            ("log_odds", FunctionTransformer(log_odds)),
            ("scale", StandardScaler()),
        ]
    )
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        (
                            "impute",
                            SimpleImputer(strategy="median", keep_empty_features=True),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            ("target_logits", encoding, categorical),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore", sparse_output=False, dtype=np.float32
                ),
                nominal,
            ),
        ]
    )
    classifier = LogisticRegression(
        C=config["logistic_c"],
        solver="newton-cholesky",
        max_iter=100,
        tol=1e-7,
        random_state=seed,
    )
    model = Pipeline([("preprocessing", preprocessing), ("classifier", classifier)])
    with threadpool_limits(limits=config["threads"]):
        model.fit(training, y)
    joblib.dump(model, path.with_suffix(".joblib"))
    details = {
        "algorithm": "logistic_regression_with_cross_fitted_target_logits",
        "C": config["logistic_c"],
        "solver": "newton-cholesky",
        "iterations_used": int(classifier.n_iter_[0]),
        "target_encoder": {
            "internal_folds": 5,
            "smooth": config["target_smoothing"],
            "unknown": "training prior",
            "log_odds": True,
        },
        "preprocessing": "Fold-local numeric imputation and scaling, cross-fitted target-encoding logits with scaling, plus one-hot encoding of original categorical features.",
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


In [ ]:
%%writefile fraudlab/refit.py
"""Refit a frozen choice once on all labeled rows, without another selection loop."""

from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from .data import fingerprint, load_data, submission, validate_submission, write_json
from .experiment import create_run, source_provenance, versions
from .features import transform_features
from .models import feature_types, fit_predict


def refit(source: Path, holdout_report: Path, output: Path, name: str) -> dict:
    started = time.perf_counter()
    provenance = source_provenance()
    config = json.loads((source / "config.json").read_text(encoding="utf-8"))
    selected = json.loads((source / "metrics.json").read_text(encoding="utf-8"))
    holdout = json.loads(holdout_report.read_text(encoding="utf-8"))
    if (
        selected.get("status") != "complete"
        or holdout.get("kind") != "independent_holdout"
        or holdout.get("status") != "complete"
    ):
        raise ValueError(
            "A completed development run and frozen independent holdout evaluation are required."
        )
    sources = [Path(path).resolve() for path in holdout["selection"]["sources"]]
    if source.resolve() not in sources:
        raise ValueError("This model was not part of the frozen holdout selection.")
    index = sources.index(source.resolve())
    if holdout["selection"]["weights"][index] <= 0:
        raise ValueError("A model with zero frozen weight does not need a final refit.")
    if (
        fingerprint(source / "metrics.json")
        != holdout["selection"]["metrics_hashes"][index]
        or fingerprint(source / "config.json")
        != holdout["selection"]["config_hashes"][index]
    ):
        raise ValueError(
            "Development metrics or configuration changed after the selection was frozen."
        )
    data_dir = Path(config["data"])
    if any(
        fingerprint(data_dir / filename) != expected
        for filename, expected in selected["data_hashes"].items()
    ):
        raise ValueError("Input data changed after development training.")
    folds = selected["model_details"]["folds"]
    if config["model"] == "mlp":
        config["epochs"] = int(
            round(float(np.median([fold["best_epoch"] for fold in folds])))
        )
    elif config["model"] in ("catboost", "lightgbm"):
        config["iterations"] = int(
            round(
                float(
                    np.median(
                        [
                            (
                                fold["trees_used"]
                                if config["model"] == "catboost"
                                else fold["best_iteration"]
                            )
                            for fold in folds
                        ]
                    )
                )
            )
        )
    config.update(
        refit=True,
        holdout_fraction=0.0,
        name=name,
        output=str(output),
        selection_source=str(source),
        holdout_report=str(holdout_report),
    )
    frame, test, sample = load_data(data_dir, config["id_column"], config["target"])
    target, id_column = config["target"], config["id_column"]
    frame = transform_features(
        frame, config.get("feature_set", "raw"), (id_column, target)
    )
    test = transform_features(
        test, config.get("feature_set", "raw"), (id_column, target)
    )
    features = selected["features"]["names"]
    run = create_run(output, name)
    write_json(run / "config.json", config)
    write_json(run / "status.json", {"status": "running"})
    try:
        _, predictions, details = fit_predict(
            config["model"],
            frame[features],
            frame[target].to_numpy(),
            test[features],
            None,
            test[features],
            config,
            config["seed"],
            run / "model",
        )
        result = submission(test[id_column], predictions, sample, target)
        result.to_csv(run / "predictions.csv", index=False)
        result.to_csv(run / "submission.csv", index=False)
        numeric, categorical = feature_types(frame[features])
        metrics = {
            "status": "complete",
            "model": config["model"],
            "phase": "full_data_refit",
            "roc_auc": None,
            "fold_scores": [],
            "fold_std": None,
            "rows": {"train": len(frame), "test": len(test), "oof": 0},
            "features": {
                "count": len(features),
                "numeric": len(numeric),
                "categorical": len(categorical),
                "names": features,
                "excluded": selected["features"]["excluded"],
            },
            "target": {"name": target, "positive_rate": float(frame[target].mean())},
            "seed": config["seed"],
            "validation": {
                "method": "no_new_selection_full_data_refit",
                "n_splits": 0,
                "early_stopping_uses_validation": False,
            },
            "selection": {
                "source_dev_run": str(source),
                "development_roc_auc": selected["roc_auc"],
                "holdout_report_sha256": fingerprint(holdout_report),
                "frozen_iterations_rule": "median best epoch/tree count across development folds; model hyperparameters unchanged",
            },
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "versions": versions(),
            "data_hashes": selected["data_hashes"],
            **provenance,
            "model_details": {
                **details,
                "feature_set": config.get("feature_set", "raw"),
            },
            "created_at": datetime.now(timezone.utc).isoformat(),
            "submission_sha256": fingerprint(run / "submission.csv"),
            "limitations": [
                "All labels were used to refit a previously frozen choice. No new independent score was measured for this final model.",
                "Wall-clock training times from concurrent runs are not controlled performance benchmarks.",
            ],
        }
        write_json(run / "metrics.json", metrics)
        write_json(run / "status.json", {"status": "complete"})
        return metrics
    except Exception as error:
        write_json(run / "status.json", {"status": "failed", "error": str(error)})
        raise


def blend_refits(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    name: str,
    id_column: str,
    target: str,
) -> dict:
    """Combine final predictions with exactly the pre-holdout frozen weights."""
    started = time.perf_counter()
    provenance = source_provenance()
    configs = [
        json.loads((run / "config.json").read_text(encoding="utf-8")) for run in runs
    ]
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    report_path = Path(configs[0]["holdout_report"])
    report = json.loads(report_path.read_text(encoding="utf-8"))
    report_hash = fingerprint(report_path)
    frozen = {
        Path(source).resolve(): weight
        for source, weight in zip(
            report["selection"]["sources"], report["selection"]["weights"]
        )
        if weight > 0
    }
    source_paths = [Path(config["selection_source"]).resolve() for config in configs]
    if set(source_paths) != set(frozen) or len(set(source_paths)) != len(source_paths):
        raise ValueError(
            "Final blend must contain every nonzero frozen source exactly once."
        )
    coefficients = np.array([frozen[source] for source in source_paths], dtype=float)
    if weights is not None:
        requested = np.array(weights, dtype=float)
        if len(requested) != len(coefficients) or not np.allclose(
            requested / requested.sum(), coefficients, rtol=0, atol=1e-12
        ):
            raise ValueError(
                "Final blend weights cannot change after independent holdout evaluation."
            )
    for config, metric in zip(configs, metrics):
        if (
            metric.get("status") != "complete"
            or metric.get("phase") != "full_data_refit"
            or metric["data_hashes"] != metrics[0]["data_hashes"]
            or metric["selection"]["holdout_report_sha256"] != report_hash
            or fingerprint(Path(config["holdout_report"])) != report_hash
        ):
            raise ValueError(
                "Final sources must be completed refits of the same frozen selection and input data."
            )
    predictions = [pd.read_csv(run / "predictions.csv") for run in runs]
    result = predictions[0].copy()
    for frame in predictions:
        validate_submission(frame, result, id_column, target)
    result[target] = sum(
        weight * frame[target].to_numpy()
        for weight, frame in zip(coefficients, predictions)
    )
    validate_submission(result, predictions[0], id_column, target)
    run = create_run(output, name)
    result.to_csv(run / "predictions.csv", index=False)
    result.to_csv(run / "submission.csv", index=False)
    details = {
        "method": "weighted_probability_average",
        "sources": [str(source) for source in runs],
        "weights": coefficients.tolist(),
    }
    metadata = {
        **metrics[0],
        **provenance,
        "model": "blend",
        "phase": "full_data_refit",
        "roc_auc": None,
        "fold_scores": [],
        "fold_std": None,
        "model_details": details,
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "submission_sha256": fingerprint(run / "submission.csv"),
        "selection": {
            "source_dev_run": report["selection"].get("source_dev_run"),
            "development_roc_auc": report["selection"].get("development_roc_auc"),
            "development_metrics_sha256": report["selection"].get(
                "development_metrics_sha256"
            ),
            "holdout_report_sha256": report_hash,
            "weights_frozen_before_holdout": True,
        },
    }
    write_json(run / "config.json", details)
    write_json(run / "metrics.json", metadata)
    write_json(run / "status.json", {"status": "complete"})
    return metadata


In [ ]:
%%writefile fraudlab/splits.py
"""Explicit split strategies shared by every model."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
    TimeSeriesSplit,
)


def make_splits(
    frame: pd.DataFrame,
    target: str,
    method: str,
    folds: int,
    seed: int,
    group_column: str | None = None,
    time_column: str | None = None,
) -> list[tuple[np.ndarray, np.ndarray]]:
    if folds < 2:
        raise ValueError("At least two folds are required.")
    y = frame[target]
    if method == "stratified":
        if y.value_counts().min() < folds:
            raise ValueError("Every class needs at least one row per fold.")
        splits = list(
            StratifiedKFold(folds, shuffle=True, random_state=seed).split(frame, y)
        )
    elif method == "group":
        if not group_column or group_column not in frame:
            raise ValueError("Group validation requires an existing --group-column.")
        groups = frame[group_column]
        if groups.isna().any() or groups.nunique() < folds:
            raise ValueError(
                "Group column must be non-null and contain at least one group per fold."
            )
        splits = list(
            StratifiedGroupKFold(folds, shuffle=True, random_state=seed).split(
                frame, y, groups
            )
        )
    elif method == "time":
        if not time_column or time_column not in frame:
            raise ValueError("Time validation requires an existing --time-column.")
        values = frame[time_column]
        if not pd.api.types.is_numeric_dtype(values):
            values = pd.to_datetime(values, errors="raise", utc=True)
        if values.isna().any():
            raise ValueError("Time column must not contain missing values.")
        # Split unique timestamps, so simultaneous events never cross a boundary.
        times = np.sort(values.unique())
        if len(times) <= folds:
            raise ValueError("Time validation needs more unique timestamps than folds.")
        splits = [
            (
                np.flatnonzero(values.isin(times[a])),
                np.flatnonzero(values.isin(times[b])),
            )
            for a, b in TimeSeriesSplit(folds).split(times)
        ]
    else:
        raise ValueError(f"Unknown validation method: {method}")
    seen: set[int] = set()
    for training, validation in splits:
        if set(training) & set(validation) or seen & set(validation):
            raise ValueError(
                "Training and validation rows must be disjoint; each OOF row is predicted once."
            )
        seen.update(validation)
        if y.iloc[training].nunique() != 2 or y.iloc[validation].nunique() != 2:
            raise ValueError(
                "Every training and validation fold needs both classes for ROC-AUC."
            )
    return splits


In [ ]:
import hashlib
import importlib.metadata as package_metadata
import json
import re
import shutil
import subprocess
import sys

SOURCE_SHA256 = {'fraudlab/__init__.py': '14479eff95a0233e8bf2d9f359843c36f53f36b92cc95c3e452f75bfb2982dcc',
 'fraudlab/__main__.py': '79e72b3a8fc1f0171b3ffc697df9c31de30451657662b74bdfb9bca7bf996668',
 'fraudlab/data.py': 'f22355cc51fa6e6162b24e639923f40a11a23af00afcb65553090176271c8326',
 'fraudlab/experiment.py': 'c5ffd6cc41aa52f639b378c187932649ca27589915c131c93666d4539a343811',
 'fraudlab/external.py': '05ec4ec64e4a2c5cf1137c949646bed14fe83fb04506b8f2f973c2c7633c5e08',
 'fraudlab/features.py': '4eb9a26b276af8c24d065e72e2672ac17edf82c09b66d9049fbe99855223ba68',
 'fraudlab/holdout.py': 'e3702891766326710c3b56dd7a7fe69f2f778acea3316665f130aa6955e2f4e3',
 'fraudlab/models.py': '0e41e75c415c7f831802cd9049e09e68c60a0685532510b32f6cb8819018730f',
 'fraudlab/refit.py': '571c4f4206b3a228341f2c5bf23a5641565e6dbf2ffe9d01ed0e7ffa12a5ce99',
 'fraudlab/splits.py': '41215280332984010f5b616bbf6a675c623b7af3e3f00604eb3238e79c709af9'}
for filename, expected in SOURCE_SHA256.items():
    assert hashlib.sha256(Path(filename).read_bytes()).hexdigest() == expected, filename

if shutil.which('git') is None:
    raise RuntimeError('A imagem precisa do executável git para registrar a proveniência; nenhum acesso remoto será feito.')

requirements = {'numpy': ((1, 24), 3), 'pandas': ((2, 0), 3),
                'scikit-learn': ((1, 3), 2), 'lightgbm': ((4, 7), 5)}
versions = {}
for package, (minimum, upper_major) in requirements.items():
    installed = package_metadata.version(package)
    version = tuple(map(int, re.findall(r'\d+', installed)[:2]))
    if version < minimum or version[0] >= upper_major:
        raise RuntimeError(f'{package} {installed} incompatível com esta revisão offline; use uma imagem Kaggle compatível.')
    versions[package] = installed
print(json.dumps(versions, indent=2))


## 2. Receita pública, inputs e créditos

A receita portátil está em `notebooks/hybrid-recipe.json` no repositório e é incorporada abaixo. Entradas: a competição S6E11 e o dataset público na **versão 2**. A referência versionada é `masayakawamata/s6e11-gpt2-target-first-oof/2`; o importador rejeita bytes diferentes do manifesto aprovado. Os arquivos são anexados como inputs, sem download pela internet durante a execução.

Previsões GPT-2: [Masaya Kawamata](https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof), CC BY 4.0. [Código de treinamento MIT](https://github.com/Matapanino/s6e11-gpt2-target-first) no commit `8446fb73c11583df4f7006aa08fa34472249b318`. Método target-first: Chris Deotte; réplica GReaT: Borisov et al.; dataset original: uditjain13. O GPT-2 foi treinado pelo autor externo; este projeto verifica e combina suas previsões públicas, sem executar seu treinamento.

Os arrays são logits float64 em ordem exata dos CSVs oficiais. O manifesto usa 10 ordens de atributos, log-sum-exp por rótulo e prior do treino de cada fold. No teste, o autor calcula a média dos cinco logits antes da sigmoid. O nosso importador aplica sigmoid e conserva autoria, versão, hashes e conversão nos metadados.

In [ ]:
RECIPE = {'protocol_id': 'hybrid5fold-public25',
 'phase': 'exploratory_full_data_cv',
 'source_revision': '30dc7eb723e4cf31291bb075ed6c0d943557264c',
 'competition': 'playground-series-s6e11',
 'rows': {'train': 699960, 'test': 299984},
 'data_hashes': {'train.csv': '87eaa10b8d17a7e4f657b8da1c66242431338bdd8b6ad45861521ac0c0e9fbd5',
                 'test.csv': '10f516959020357b565abaaa4fd25baae97fc2f536d0805695bb7cc15c1660f0',
                 'sample_submission.csv': '13fc6ed8b73de5329884ec490f2a0b750cf29a7cda772b78cfa0cf486f6f35fa'},
 'common': {'phase': 'exploratory_full_data_cv',
            'feature_set': 'value_digits',
            'id_column': 'id',
            'target': 'is_fraud',
            'split': 'stratified',
            'folds': 5,
            'seed': 42,
            'holdout_fraction': 0.0,
            'fixed_training': True,
            'validation_note': 'Exploratory full-data five-fold CV aligned with approved public '
                               'GPT-2 release2. All labels, including the historical holdout, '
                               'participate in fold training. No independent holdout remains for '
                               'this stage. Own model settings are frozen; no new hyperparameter '
                               'selection.'},
 'models': {'glm': {'name': 'explore5-glm',
                    'model': 'glm',
                    'threads': 4,
                    'device': 'auto',
                    'logistic_c': 0.3,
                    'target_smoothing': 50.0},
            'lightgbm': {'name': 'explore5-lgbm',
                         'model': 'lightgbm',
                         'threads': 6,
                         'device': 'auto',
                         'iterations': 1438,
                         'learning_rate': 0.04,
                         'l2_leaf_reg': 5.0,
                         'num_leaves': 4,
                         'min_child_samples': 100,
                         'target_smoothing': 20.0,
                         'patience': 150}},
 'own_weights': {'glm': 0.75, 'lightgbm': 0.25},
 'hybrid_weights': {'own': 0.75, 'external_gpt2': 0.25},
 'effective_weights': {'glm': 0.5625, 'lightgbm': 0.1875, 'external_gpt2': 0.25},
 'public_component': {'name': 'GPT-2 target-first',
                      'kind': 'gpt2_target_first',
                      'source_type': 'external_predictions',
                      'trained_by': 'Masaya Kawamata',
                      'model_architecture': 'GPT-2 target-first classifier',
                      'training_code_url': 'https://github.com/Matapanino/s6e11-gpt2-target-first',
                      'training_code_commit': '8446fb73c11583df4f7006aa08fa34472249b318',
                      'code_license': 'MIT',
                      'predictions_source_url': 'https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof',
                      'predictions_source_version': 2,
                      'predictions_download_ref': 'masayakawamata/s6e11-gpt2-target-first-oof/2',
                      'predictions_license': 'CC BY 4.0',
                      'data_license': 'CC BY 4.0 (prediction dataset)',
                      'method_credits': ['Chris Deotte: target-first GPT-2 method',
                                         'Borisov et al.: GReaT / Language Models are Realistic '
                                         'Tabular Data Generators',
                                         'uditjain13: original Credit Card Fraud Detection 2026 '
                                         'dataset'],
                      'training_executed_locally': False,
                      'trained_by_us': False,
                      'oof_auc': 0.9500272037932734,
                      'validation': {'method': 'StratifiedKFold',
                                     'n_splits': 5,
                                     'shuffle': True,
                                     'seed': 42},
                      'inference': {'method': 'target-first label likelihood ratio',
                                    'target_first': True,
                                    'likelihood_ratio': True,
                                    'prior': "training rate of the fold's fit rows",
                                    'orders': 10,
                                    'order_aggregation': 'log-sum-exp over orders per label, then '
                                                         'the difference, plus the prior',
                                    'test_aggregation': 'mean of fold log-odds',
                                    'test_ensemble_folds': 5,
                                    'source_test_aggregation': 'mean over fold models of the '
                                                               'per-fold log-odds',
                                    'import_transform': 'sigmoid(log_odds), float64'},
                      'weight': 0.25,
                      'mixture_space': 'probability',
                      'manifest_sha256': '0083d23b46f71311ceeed66874b093ec927d25d98b37f6d2e78b1de17f0f22d0',
                      'array_sha256': {'oof.npy': '3468aaaeb7851739ec0de1e06b780da6d2777fcaeef638a6852c56e68e67a206',
                                       'test.npy': '6c27aff7be744f7a0cc17809e0cd04fcdea0664234af88e3040a25235fb67906',
                                       'fold_ids.npy': '1d459900f6c3a001ff971c09d9c607b383bdc732105c54701587189e52eeab45'}},
 'observed_local_results': {'own_oof_auc': 0.9506227499782469,
                            'hybrid_oof_auc': 0.9512586315198723,
                            'scope': 'Recorded local exploratory OOF; not outputs of a saved, '
                                     'unexecuted Kaggle notebook.'},
 'historical_protocol': {'source_revision': '1fe34b540f3ea28276da652c3dd1de02726897f4',
                         'independent_holdout_applies_to_hybrid': False,
                         'regenerate_argument': '--protocol artifacts/holdout/evaluation.json',
                         'note': 'The earlier 3-fold/15-percent holdout protocol is historical '
                                 'only; no independent holdout remains for the hybrid.'},
 'output': 'submission.csv'}
candidates = [Path('/kaggle/input/competitions/playground-series-s6e11'),
              Path('/kaggle/input/playground-series-s6e11')]
data = next((path for path in candidates if (path / 'train.csv').is_file()), None)
if data is None:
    raise FileNotFoundError('Adicione playground-series-s6e11 aos inputs.')
for filename, expected in RECIPE['data_hashes'].items():
    if hashlib.sha256((data / filename).read_bytes()).hexdigest() != expected:
        raise ValueError(f'{filename}: versão diferente do protocolo documentado.')
manifest_hash = RECIPE['public_component']['manifest_sha256']
public_source = next((path.parent for path in Path('/kaggle/input').rglob('manifest.json')
                      if hashlib.sha256(path.read_bytes()).hexdigest() == manifest_hash), None)
if public_source is None:
    raise FileNotFoundError('Adicione o dataset GPT-2 versão 2; o manifesto deve coincidir com a receita.')

runs = Path('artifacts/runs')
def run(*arguments):
    subprocess.run([sys.executable, '-m', 'fraudlab', *map(str, arguments)], check=True)

run('inspect', '--data', data, '--output', 'artifacts/profile.json')
public_run = runs / 'explore5-public-gpt2'
run('import-public-gpt2', '--source', public_source, '--data', data,
    '--output', runs, '--name', public_run.name)


## 3. Treinos próprios fixos em 5/42

Todas as 699.960 linhas participam da CV; cada previsão OOF vem do modelo que excluiu seu fold. GLM: C=0,3 e target encoding interno com suavização 50. LGB: 1.438 árvores fixas, 4 folhas, taxa 0,04, L2=5 e suavização 20. Atributos e demais parâmetros permanecem congelados. Não há early stopping, nova escolha de árvores ou reserva independente.

Os treinos são sequenciais para limitar RAM. O teste de cada família recebe a média de probabilidades dos cinco modelos. Essas previsões são diferentes do refit único do protocolo histórico; os registros mantêm os protocolos separados.

In [ ]:
def train_model(model):
    configuration = {'data': str(data), 'output': str(runs),
                     **RECIPE['common'], **RECIPE['models'][model]}
    arguments = ['train']
    for key, value in configuration.items():
        if value is None or value is False:
            continue
        arguments.append('--' + key.replace('_', '-'))
        if value is not True:
            arguments.extend(map(str, value if isinstance(value, list) else [value]))
    run(*arguments)
    return runs / configuration['name']

glm_run = train_model('glm')
lgbm_run = train_model('lightgbm')


## 4. Composição congelada e arquivo de submissão

A composição própria usa 75% GLM e 25% LGB. Ela recebe 75% do peso total; o GPT-2 público recebe 25%. Pesos efetivos: **56,25% GLM / 18,75% LGB / 25% GPT-2 externo**, em probabilidades. Não há ajuste pelos scores do Kaggle.

A grade local predefinida testou 10/25/50/75% de GPT-2. A opção de 25% melhorou os cinco folds; 25% e 50% ficaram praticamente empatadas. Mantivemos 25% para preservar maior contribuição própria com o mesmo ganho observado. Esses resultados são seleção exploratória por OOF, não uma avaliação independente; esta versão salva não os executou no Kaggle.

In [ ]:
own_run = runs / 'explore5-own'
hybrid_run = runs / 'explore5-public25'
run('blend', '--runs', glm_run, lgbm_run,
    '--weights', RECIPE['own_weights']['glm'], RECIPE['own_weights']['lightgbm'],
    '--output', runs, '--name', own_run.name)
run('blend', '--runs', own_run, public_run,
    '--weights', RECIPE['hybrid_weights']['own'], RECIPE['hybrid_weights']['external_gpt2'],
    '--output', runs, '--name', hybrid_run.name)
final_csv = hybrid_run / 'submission.csv'
run('validate-submission', '--file', final_csv, '--sample', data / 'sample_submission.csv')
shutil.copyfile(final_csv, 'submission.csv')
metrics = json.loads((hybrid_run / 'metrics.json').read_text())
assert metrics['phase'] == 'exploratory_full_data_cv'
assert metrics['validation']['independent_holdout'] is False
assert metrics['external_components'][0]['trained_by_us'] is False
assert metrics['external_components'][0]['weight'] == RECIPE['hybrid_weights']['external_gpt2']
assert hashlib.sha256(Path('submission.csv').read_bytes()).hexdigest() == metrics['submission_sha256']
print({'file': 'submission.csv', 'sha256': metrics['submission_sha256'],
       'exploratory_oof_auc': metrics['roc_auc'], 'external_components': metrics['external_components']})


## Histórico e limites da reprodução

O protocolo próprio anterior usou 3 folds e uma reserva de 15%, avaliada uma vez antes de refits em 100%. Ele pode ser regenerado pelo mesmo script com `--protocol artifacts/holdout/evaluation.json` ou consultado na história Git. **Seu holdout não é independente desta composição híbrida**: a CV nova e o componente público usam todos os rótulos. O notebook canônico não chama a avaliação dessa reserva novamente.

OOF exploratória local registrada: 0.95125863. Scores e posições realmente processados pelo Kaggle constam nas evidências do repositório; não são outputs deste notebook salvo sem execução. Diferenças de bibliotecas/hardware podem mudar a reprodução. Nenhuma submissão é enviada automaticamente, e a classificação privada final só existe após o encerramento.

## Continuação de 17/11/2026 — fonte pública e avaliações

As células acima preservam a reprodução da receita v0.1 e seus módulos congelados. A implementação integrada dos experimentos posteriores está no [GitHub público, commit 6ca52c7](https://github.com/aikolopes-DS/fraudlab/tree/6ca52c7029d32f9e2a860a752f6b139ebbc03654). Esse código foi testado localmente e em CI; esta versão do notebook foi salva **sem execução na nuvem**.

As duas avaliações seguintes usam arquivos e pesos congelados no [plano público](https://github.com/aikolopes-DS/fraudlab/blob/6fda338/evidence/public-evaluation-plan.json): refits próprios em 100% com o mesmo GPT-2 v2 aprovado; e GLM com chaves BPE e frequência, junto aos componentes anteriores. O [compositor de refits](https://github.com/aikolopes-DS/fraudlab/blob/6ca52c7/scripts/compose_refit_hybrid.py) gera somente inferência, sem nova AUC ou OOF. Os artefatos históricos e seus hashes são necessários para reproduzir esse arquivo.

A mistura BPE/frequência alcançou OOF 0,95140705, mas melhorou somente três folds e falhou no critério anterior de quatro. Sua avaliação pública é uma **exceção exploratória documentada**, não uma promoção por aquele critério nem comprovação de melhoria robusta. Não há holdout independente nesta continuação. Treinos, resultados negativos e créditos estão no repositório; os scores processados e a posição provisória constam do [leaderboard oficial](https://www.kaggle.com/competitions/playground-series-s6e11/leaderboard) e da [evidência atual](https://github.com/aikolopes-DS/fraudlab/blob/main/evidence/public.json).

Masaya Kawamata treinou o GPT-2 externo: previsões v2 sob CC BY 4.0, código externo MIT. O projeto não reexecutou esse treinamento. TabM própria foi treinada e comparada, mas descartada da composição final. Nenhum array da fonte seed2026 não aprovada foi integrado.
